# Markov / stochastic games — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def expected_payoff(A,p,q): return float(np.asarray(p) @ np.asarray(A) @ np.asarray(q))
def best_response_payoffs(A,q): return np.asarray(A) @ np.asarray(q)
print('setup ok')

## Value iteration in a two-state stochastic game
Stochastic games combine Markov state transitions with strategic actions. These examples compute one Bellman backup, iterate values, expose policy coupling, compare discount factors, and visualize a tiny equilibrium policy over states.

In [ ]:
# 1) One Bellman backup for a fixed joint policy
# state 0 reward 1 then stay with .8; state 1 reward 0 then return with .3, gamma=.9
V=np.array([0.,0.]); gamma=.9; P=np.array([[.8,.2],[.3,.7]]); r=np.array([1.,0.])
backup=r+gamma*P@V
plt.figure(figsize=(4,3)); plt.bar(['s0','s1'],backup); plt.title('first backup equals immediate reward')
assert np.allclose(backup,[1,0])

In [ ]:
# 2) Value iteration converges for the fixed policy
gamma=.9; P=np.array([[.8,.2],[.3,.7]]); r=np.array([1.,0.]); V=np.zeros(2); hist=[]
for _ in range(120): V=r+gamma*P@V; hist.append(V.copy())
hist=np.array(hist)
plt.figure(figsize=(5,3)); plt.plot(hist[:,0],label='V(s0)'); plt.plot(hist[:,1],label='V(s1)'); plt.legend(); plt.title('values converge under discounting')
assert abs(V[0]-6.727)<0.01 and abs(V[1]-4.909)<0.01

In [ ]:
# 3) A player's best action depends on the other player's action
# state s0: row payoff matrix for two joint actions
A=np.array([[2,0],[0,1]])
q=np.array([.25,.75]); pay=A@q
plt.figure(figsize=(5,3)); plt.bar(['row action 0','row action 1'],pay); plt.title('best response changes with opponent mix q')
assert np.allclose(pay,[0.5,0.75]) and int(np.argmax(pay))==1

In [ ]:
# 4) Larger discount makes future state transitions matter more
P=np.array([[.8,.2],[.3,.7]]); r=np.array([1.,0.])
def solve(g): return np.linalg.solve(np.eye(2)-g*P,r)
gammas=np.array([.1,.5,.9,.99]); vals=np.array([solve(g)[0] for g in gammas])
plt.figure(figsize=(5,3)); plt.plot(gammas,vals,'-o'); plt.xlabel('gamma'); plt.ylabel('V(s0)'); plt.title('patient agents value long-run rewards more')
assert vals[-1]>vals[0] and abs(solve(.9)[0]-6.727)<0.01

In [ ]:
# 5) Tiny state-dependent equilibrium policy heatmap
# s0 uses mixed coordination p=.6 for action 0; s1 uses p=.2
policy=np.array([[.6,.4],[.2,.8]])
plt.figure(figsize=(4,3)); plt.imshow(policy,cmap='viridis',vmin=0,vmax=1); plt.xticks([0,1],['a0','a1']); plt.yticks([0,1],['s0','s1']); plt.colorbar(label='probability'); plt.title('policies are functions of state')
assert np.allclose(policy.sum(axis=1),[1,1]) and policy[0,0]>policy[1,0]